# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We walk through Croissant-powered data discovery, loading, preview, EDA, and visualization using the unique `@id` references as required for robust, FAIR exploration.

### Dataset Source
This dataset is defined by its [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
# Access metadata as object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all record sets in the dataset. For each, we show their `@id`, name, and include the list of field `@id`s and field names. All identifiers are direct `@id` keys from the schema.

In [ ]:
# List record sets by `@id` and their fields by `@id`
recordsets = dataset.record_sets

print(f"Record sets found: {len(recordsets)}\n")

recordsets_summary = []
for rs in recordsets:
    summary = {
        '@id': rs.id,
        'name': rs.name,
        'fields': [(field.id, field.name) for field in rs.fields]
    }
    recordsets_summary.append(summary)
    print(f"RecordSet: {rs.name} (id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id})")
    print()

# Store record set `@id`s for subsequent steps
record_set_ids = [rs['@id'] for rs in recordsets_summary]
if len(record_set_ids):
    print(f"RecordSet @ids: {record_set_ids}")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract all record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for the record set by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")
    dataframes[record_set_id] = df

# Display first dataframe's columns as example
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All references use `@id` identifiers.

Below, we pick the first available record set and try to identify numeric fields to demonstrate EDA steps like filtering, normalization, and grouping.

In [ ]:
# Select a record set
if not record_set_ids:
    print("No record set found in this dataset to perform EDA.")
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Attempt to find numeric field candidates by data type or column names
    # We'll pick any field with numeric dtype, or commonly numeric labels
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    # If no inferred numeric field, try to guess by column name
    if not numeric_fields:
        numeric_fields = [col for col in df.columns if any(term in col.lower() for term in ["age", "count", "interval", "years", "score", "duration"])) ]
    if not numeric_fields:
        numeric_fields = [df.columns[0]]  # Fallback to first column

    print(f"Numeric fields available: {numeric_fields}")
    numeric_field = numeric_fields[0]

    # Set an arbitrary threshold for filtering, if possible
    # Try 10 if the field seems to be age or interval, otherwise use median
    try:
        threshold = 10 if ("age" in numeric_field.lower() or "interval" in numeric_field.lower()) else df[numeric_field].median()
    except Exception:
        threshold = 0

    try:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize this numeric field
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, col_norm]].head())
        
    except Exception as e:
        print(f"Could not filter or normalize: {e}")

    # Try grouping on a likely categorical field
    group_field_candidates = [col for col in df.columns if col != numeric_field and (df[col].dtype == object or df[col].dtype == 'category')]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        try:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        except Exception as e:
            print(f"Could not group: {e}")
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using Matplotlib or Seaborn as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field, and optionally its normalized version
if record_set_ids:
    fig, ax = plt.subplots(figsize=(7, 4))
    if 'col_norm' in locals() and col_norm in filtered_df.columns:
        sns.histplot(filtered_df[col_norm], kde=True, color='orange', ax=ax)
        ax.set_title(f"Distribution of normalized {numeric_field} (Filtered)")
        ax.set_xlabel(f"{numeric_field} (normalized)")
    else:
        sns.histplot(filtered_df[numeric_field], kde=True, ax=ax)
        ax.set_title(f"Distribution of {numeric_field} (Filtered)")
        ax.set_xlabel(f"{numeric_field}")
    plt.tight_layout()
    plt.show()

    # Optional: scatter plot if grouped by a categorical variable
    if 'group_field' in locals():
        fig, ax = plt.subplots(figsize=(7, 4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field], ax=ax)
        ax.set_title(f"{numeric_field} by {group_field}")
        ax.set_xlabel(group_field)
        ax.set_ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² dataset using the Croissant schema for reproducible tabular data access in Python. We:
- Inspected metadata and listed all record sets and fields by `@id` as per the schema.
- Loaded records into dataframes, filtered, normalized, and grouped data using field `@id` references.
- Created summary visualizations for the inspected record set.

You may extend this analysis by referencing additional record sets or fields (by their `@id`), enriching processing steps, or building custom downstream ML workflows.